In [1]:
# pip install -q youtube-transcript-api langchain-community langchain-openai \faiss-cpu tiktoken python-dotenv

# 1. Environment & Package Installation

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

video_id="LOUaom9HZIg"
try:
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id, languages=["en"])

    # 3. Combine the text blocks
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("Error: Subtitles/transcripts are disabled for this video.")




hello guys in this video we are going to create an amazing endtoend document Q&A chatbot using Google open source model that is called as gamma now if you don't know about Gamma Gamma is a family of lightweight state-of Art open- Source models built from the same research and technology used to create the Gemini models now you should understand one thing that they are bigger bigger companies like meta there are uh companies like Google who are specifically working on open source models like glama Lama 3 so why Google will be behind that right so it'll obviously create its own open source model and that is where gamma and its variant models are there so uh in this video I'll show you how you can actually use Gemma completely from scratch we will also be using one amazing inferencing engine which is called as Gro so I hope everybody has heard about this particular Gro okay if you have not heard don't worry we will be understanding about this why grock will be used uh specifically if you 

# 2. Data Ingestion: Fetch YouTube Transcript
Fetch the official or auto-generated English captions from the target YouTube video using its `video_id`. The individual transcript text blocks are concatenated into a single continuous text string.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_split=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=70,separators=["\n\n", "\n", " ", ""])
chunk=text_split.create_documents([transcript])

print(len(chunk))

68


# 3. Text Chunking & Splitting

Split the raw transcript into smaller, manageable chunks using `RecursiveCharacterTextSplitter`. 

- **`chunk_size`**: 500 characters per chunk.
- **`chunk_overlap`**: 70 characters overlap to retain context across boundaries.
- **`separators`**: Split recursively on double newlines, single newlines, spaces, and empty characters.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 4. Initialize Embedding Model

Load the open-source **`sentence-transformers/all-MiniLM-L6-v2`** embedding model via `HuggingFaceEmbeddings` to transform textual chunks into dense numerical vectors.

In [5]:
from langchain_community.vectorstores import FAISS

vector_store=FAISS.from_documents(
    embedding=embedding_model,
    documents=chunk
)

print(vector_store.index.ntotal)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_23676\446407348.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


68


# 5. Index Chunks in FAISS Vector Store

Generate vector representations for all transcript chunks and index them into an in-memory **FAISS** vector store for fast similarity search retrieval.

In [6]:
retrieve=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})
query="Which exact Google Generative AI embedding model variant name is specified in the code to convert text chunks into vectors?"
retriever=retrieve.invoke(query)
print(query)
for i,doc in enumerate(retriever):
    print(doc.page_content)

Which exact Google Generative AI embedding model variant name is specified in the code to convert text chunks into vectors?
because our main aim is that we'll be reading some PDF files uh from a folder and uh then we will be reading all the documents and then we will be dividing Chun into chunks be uh using this recursive character text plater and finally we are also going to use one more library that is our embedding technique and for that we are going to import from Lang chain _ Google gen so Google already provides this Google generative AI embedding which you can completely use it completely for free just by
this F f.r documents and here we are going to use this store st. sore state. final documents and the embedding techniques will be the same and embedding technique that we have initialized over here that is nothing but Google generative AI embedding right and this embedding will be responsible in converting all these final documents into vectors and this F will be responsible in

# 6. Similarity Search Retrieval Test

Convert the FAISS index into a retriever (`k=4`) and test query execution to verify that relevant transcript chunks are successfully fetched based on semantic similarity.

In [7]:
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate.from_template(
    """ You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

Context:
{context}

Question:
{question}

Answer:
"""
)

# 7. Define RAG Prompt Template

Construct a strict system prompt using `PromptTemplate`. The prompt forces the LLM to answer questions **only** based on the provided transcript context and explicitly disallows hallucinations.

In [8]:
from langchain_huggingface import HuggingFacePipeline

llm=HuggingFacePipeline.from_model_id(
    model_id="google/flan-t5-large",
    task="text2text-generation",
    pipeline_kwargs={"max_new_tokens":100,"do_sample":False}
)

print("Language Model Initiated")

Device set to use cpu


Language Model Initiated


# 8. Load HuggingFace Pipeline LLM

Initialize the open-source **`google/flan-t5-large`** model locally using `HuggingFacePipeline` for downstream text generation.

In [9]:
query="Which exact Google Generative AI embedding model variant name is specified in the code to convert text chunks into vectors?"

result=llm.invoke(query)
print(result)

GM


# 10. Construct Modern LCEL RAG Chain

Assemble the complete Retrieval-Augmented Generation pipeline using modern **LangChain Expression Language (LCEL)** components:

- **`RunnableParallel`**: Parallelizes document retrieval (formatted via `RunnableLambda`) and user query passthrough.
- **`rag_chain`**: Sequential execution piping `parallel_chain` $\rightarrow$ `prompt` $\rightarrow$ `llm` $\rightarrow$ `StrOutputParser`.

In [10]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

parallel_chain = RunnableParallel({
    'context': retrieve | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

rag_chain=parallel_chain | prompt | llm | StrOutputParser()


# 11. Execute RAG Query

Invoke the end-to-end `rag_chain` with a specific target query to retrieve grounded context and synthesize a precise answer.

In [33]:
query="What text splitter strategy and parameters are used to chunk documents in the video?"

In [34]:
result=rag_chain.invoke(query)
print(f"Question: {query}")
print(f"Answer: {result}")

Question: What text splitter strategy and parameters are used to chunk documents in the video?
Answer: recursive character text splitter and chunk size of 1,000 and chunk overlap of 200
